<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Dataset anatomy

**[Dataset anatomy](https://learning.nextia-ai.com/courses/sql/m01/dataset-anatomy/)** · SQL and Data Preparation for AI · Module 1, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read a dataset before you query it: rows, columns, types, identifiers and timestamps. You answer the most important question about each table: what does one row mean?

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | How to read a CSV file and run Python code, from [Python for Practical AI Engineering](https://learning.nextia-ai.com/courses/python/). You do not need to know SQL. |
| **You do not need** | Anything on your computer. The setup below builds the practice database here. |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m01/dataset-anatomy/) has the full explanations and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your query between the triple quotes `"""`, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M01-L01-dataset-anatomy/dataset-anatomy-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The help-desk export

Larkfield is a fictional online shop for home and garden products. When a customer has a problem, the help-desk system records a **ticket**. Larkfield wants to know, when a ticket arrives, if it will need a senior agent within 72 hours: an **escalation**.

Mei, the data engineer, gets an **export**: a copy of the help-desk data, written to files at one moment. It has three CSV files: `customers.csv`, `tickets.csv` and `outcomes.csv`. The setup cell loaded them into one **database**, `larkfield.db`. The database is **SQLite**, which keeps the whole database in one file. The data is made up for this course; it has no real people.

## 2. Rows, columns and values on four tickets

A **query** is a question that you ask the database, written in **SQL**. The next query shows four tickets, with five of their columns.

> **Predict.** One of the four tickets came from a guest who bought without an account. What do you expect in its `customer_id`? Then run the cell.

In [ ]:
sql("""
SELECT ticket_id, customer_id, created_at, channel, priority
FROM tickets
WHERE ticket_id IN ('T-10001', 'T-10002', 'T-10003', 'T-10064');
""")

> **Check.** You should see 4 rows. T-10064 has `NULL` in `customer_id`: no value.

- A **table** is a set of data in rows and columns, with a name. This one is `tickets`.
- A **row** is one line of the table. Here, each row describes one ticket. ("Record" and "tuple" mean the same as row.)
- A **column** has a name and holds one kind of fact for every row, for example `channel`.
- A **value** is what one row has in one column: the `channel` of T-10003 is `Email`.
- **NULL** means "no value". It is not zero and not an empty text.

Look at T-10003: its channel is `Email` with a capital E. For a computer, `email` and `Email` are two different values. Here you only notice it; Module 2 comes back to it.

## 3. Types, identifiers and timestamps

Each column has a **type**: the kind of value that it holds. SQLite uses `TEXT`, `INTEGER` (whole numbers) and `REAL` (numbers with a decimal part). The SQL function `typeof()` tells you the type of one value. Run the cell.

In [ ]:
sql("""
SELECT ticket_id, typeof(ticket_id), priority, typeof(priority), order_value, typeof(order_value)
FROM tickets
WHERE ticket_id = 'T-10001';
""")

> **Check.** You should see `T-10001` as `text`, `2` as `integer` and `47.25` as `real`.

The description of all tables, their columns and their types is the **schema**. The schema tells you what the data can hold. Only the data tells you what it actually holds.

- An **identifier** names one thing, such as `T-10001` for a ticket. You do not calculate with identifiers. You use them to find a thing, and to connect a row in one table to a row in another.
- A **timestamp** says when something happened, for example `created_at`. SQLite has no date type: the export writes timestamps as text, `YYYY-MM-DD HH:MM:SS`. In this form, the order of the text is also the order in time.

In AI work, a timestamp also tells you **when a value was known**. A model that predicts at ticket creation can use only what was known at that moment. Module 5 builds on this.

## 4. Run your first query

The next query asks for all columns (`*`) of `tickets`, but only the first 5 rows (`LIMIT 5`).

> **Predict.** How many columns does `tickets` have? Then run the cell.

In [ ]:
sql("""
SELECT *
FROM tickets
LIMIT 5;
""")

> **Check.** You should see 5 rows and 11 columns. The first row is T-10001. Then change `5` to `10` in the cell, and run it again.

Some columns need the person who knows the help desk. Grace, the support lead, explains three of them:

- `priority` is set when the ticket arrives: 1 is the most urgent, 3 the least.
- `priority_now` is the priority today. It can change after creation.
- `closed_at` is when the ticket was closed. It is NULL when the ticket is still open.

## 5. The question that comes first: what does one row mean?

The **grain** of a table is what one row represents. Ask this question before any count. Look at a small sample of each table. Run the three cells.

In [ ]:
sql("""
SELECT customer_id, segment, region, joined_on
FROM customers
LIMIT 3;
""")

> **Check.** You should see C-0001, C-0002 and C-0003. One row is one **customer**.

Customer C-0001 has three tickets. How many rows does C-0001 have in `tickets`?

In [ ]:
sql("""
SELECT ticket_id, customer_id, created_at, team
FROM tickets
WHERE customer_id = 'C-0001';
""")

> **Check.** You should see 3 rows: T-10270, T-10530 and T-10714. One row is one **ticket**, so C-0001 appears in three rows.

Ticket T-10014 was resolved, then reopened, then resolved again.

In [ ]:
sql("""
SELECT outcome_id, ticket_id, recorded_at, status
FROM outcomes
WHERE ticket_id = 'T-10014';
""")

> **Check.** You should see 3 rows: `resolved`, `reopened` and `resolved`. One row is one **event** in the life of a ticket.

The three tables have three different grains:

| Table | One row means | Identifier of the row |
|---|---|---|
| `customers` | one customer | `customer_id` |
| `tickets` | one ticket | `ticket_id` |
| `outcomes` | one event in a ticket's life | `outcome_id` |

A count of rows in `outcomes` is a count of events, not of tickets.

## 6. The snapshot

The export is a **snapshot**: the state of the data at one fixed moment. Everything that happened after that moment is not in it. The table `snapshot` records the moment. Run the cell.

In [ ]:
sql("""
SELECT *
FROM snapshot;
""")

> **Check.** You should see 3 rows. The row `exported_at` has the value `2026-07-01 06:00:00`.

The snapshot explains some values. Count the tickets that have no `closed_at`. Run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS open_ticket_rows
FROM tickets
WHERE closed_at IS NULL;
""")

> **Check.** You should see **12**. These tickets were still open at 06:00 on 1 July. A later export can have a value for them. So write the snapshot time next to every number that you report: "1,417 ticket rows in the export of 2026-07-01 06:00".

## 7. Worked example: how big is each table?

Priya asks Mei: "How much data is there?" `COUNT(*)` counts rows.

> **Predict.** Which table has the most rows? Then run the cell.

In [ ]:
sql("SELECT COUNT(*) AS customer_rows FROM customers;")
sql("SELECT COUNT(*) AS ticket_rows FROM tickets;")
sql("SELECT COUNT(*) AS outcome_rows FROM outcomes;")

> **Check.** You should see **240**, **1417** and **1874**.

Mei reads each number against the grain:

- **240 customers.** One row is one customer.
- **1,417 ticket rows.** One row should be one ticket. Mei writes "ticket rows", not "tickets", until she checks that no ticket appears twice. You do that check in the next lesson.
- **1,874 outcome rows.** One row is one event, and a ticket can have several events. So 1,874 is not a number of tickets.

## 8. Guided practice: look at the order of the events

Run the cell. Look at the `ticket_id` column.

In [ ]:
sql("""
SELECT *
FROM outcomes
LIMIT 6;
""")

> **Check.** You should see 6 rows. The rows are in the order the events were recorded, not in ticket order: T-10002 comes first.

## 9. Your turn: one customer's tickets

> **Your turn.** The starting query shows every event of ticket T-10014. Change it so that it shows **every column** of every **ticket** of customer **C-0001**. Predict the number of rows first. Text values need single quotes: `'C-0001'`. Then run the check cell. When it is correct, you see `Check passed.`

In [ ]:
customer_tickets = """
SELECT *
FROM tickets
WHERE customer_id = 'C-0001';
"""
sql(customer_tickets)

In [ ]:
check(customer_tickets, rows=3, columns=11, expected="313768d529419f2e")

The grain tells you which table to use. "Tickets of a customer" are rows in `tickets`. "Events of a ticket" are rows in `outcomes`.

## 10. Failure case: counting events as tickets

Omar, the product manager, asks how many tickets were resolved. Tomás counts the `resolved` rows.

> **Predict.** Can the answer be larger than the number of different tickets, 1,403? Then run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS resolved_rows
FROM outcomes
WHERE status = 'resolved';
""")

> **Check.** You should see **1533**. That is more than the 1,403 different tickets in the export, so it cannot be a count of tickets. The query gave no error.

**Cause:** the grain of `outcomes` is one event. A ticket that was resolved, reopened and resolved again, like T-10014, has two `resolved` rows.

> **Your turn.** Count each ticket once. `COUNT(DISTINCT ticket_id)` counts the different ticket IDs (Module 2 teaches it). Change the query so that it counts the different tickets with a `resolved` event. Then run the check cell.

In [ ]:
resolved_tickets = """
SELECT COUNT(DISTINCT ticket_id) AS resolved_tickets
FROM outcomes
WHERE status = 'resolved';
"""
sql(resolved_tickets)

In [ ]:
check(resolved_tickets, rows=1, columns=1, expected="964dfef7a99a56ab")

The result is **1,396** tickets. Before you count, say the grain of the table aloud: "one row is one event".

**In AI work:** the same mistake gives a training table with one row per event instead of one row per ticket. Tickets with many events then count several times, and the model learns more from them.

## Check your understanding and recap

Answer the three **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m01/dataset-anatomy/). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that a table has rows and columns, that each column has a type, and that NULL means "no value". The **grain** says what one row means: one customer, one ticket, or one event in a ticket's life. Say it before you count. The export is a **snapshot** of 2026-07-01 06:00: write that time next to your numbers.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Relational concepts](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m01/dataset-anatomy/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.